# Task 8 — HYDE 3.5: Per-Epoch Marginal Distributions and Temporal Structure

HYDE 3.5 (Klein Goldewijk et al.) reconstructs global land use and population from
10,000 BCE to 2023 CE at 5-arc-minute resolution. Unlike the static Band A–E variables
(one value per basin, fixed) and eVolv2k (a sparse event catalog), HYDE is a dense
gridded dataset where the value at each cell changes at every time step.

**Five variables in hand** (all as NetCDF):
- `cropland.nc` — fraction of cell under cultivation
- `grazing_land.nc` — fraction under grazing
- `urban_area.nc` — fraction urbanized
- `population_density.nc` — persons per km²
- `total_rice.nc` — fraction under rice cultivation

**Approach**: inspect structure first, then characterize distributions across
representative epochs. The data drives the cell design — do not assume time
encoding, units, or fill values before reading.

Outputs: `output/edop/explore/08_*.{csv,png}`
Findings: `logs/exploration_log.md` under Task 8.

In [1]:
# Cell 1 — Structural inspection: open one file, understand the dataset before anything else
#
# Questions to answer:
#   - What are the dimensions and their sizes?
#   - What does the time axis actually contain? (years? indices? calendar?)
#   - What are the variable names inside the file?
#   - What fill/missing value is used?
#   - What units are reported in the metadata?
#   - How large is the spatial grid?

import numpy as np
import xarray as xr
from pathlib import Path

HYDE = Path('/Users/karlg/Documents/Repos/_cedop/data/hyde/NetCDF')
OUT  = Path('/Users/karlg/Documents/Repos/_cedop/output/edop/explore')
OUT.mkdir(parents=True, exist_ok=True)

# Open cropland as the probe file — lazy load, no compute yet
ds = xr.open_dataset(HYDE / 'cropland.nc')

print('=== Dataset structure ===')
print(ds)
print()
print('=== Dimensions ===')
for dim, size in ds.dims.items():
    print(f'  {dim}: {size}')
print()
print('=== Variables ===')
for var in ds.data_vars:
    v = ds[var]
    print(f'  {var}: shape={v.shape}, dtype={v.dtype}')
    print(f'    attrs: {dict(v.attrs)}')
print()
print('=== Time axis (first 10 and last 10 values) ===')
if 'time' in ds.coords:
    t = ds['time'].values
    print(f'  dtype: {t.dtype}')
    print(f'  length: {len(t)}')
    print(f'  first 10: {t[:10]}')
    print(f'  last 10:  {t[-10:]}')
else:
    print('  No "time" coordinate — check dimension names above')
print()
print('=== Spatial coordinates ===')
for coord in ['lat', 'lon', 'latitude', 'longitude']:
    if coord in ds.coords:
        c = ds[coord].values
        print(f'  {coord}: {len(c)} values, {c.min():.3f} to {c.max():.3f}, step~{abs(c[1]-c[0]):.4f}°')
print()
print('=== Global attributes ===')
for k, v in ds.attrs.items():
    print(f'  {k}: {v}')

/var/folders/w1/ms_2x6rj0ls88v79q33lvds80000gp/T/ipykernel_27974/340159854.py:26: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():
<frozen _collections_abc>:894: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.


In [2]:
# Cell 2 — Time step inventory and multi-file variable check
#
# Two goals:
#   (a) Print all 128 time steps as year integers so we can see the
#       resolution structure and choose actual available epochs.
#   (b) Open all 5 files and report variable name + units for each —
#       do not assume they match cropland.
#
# Note: cftime uses has_year_zero=True (astronomical numbering).
# Year 0 = 1 BCE. Use t.year directly; negative = BCE.

# (a) Extract year integers from the time axis
time_years = [t.year for t in ds['time'].values]
print('=== All 128 time steps (year, astronomical) ===')
print('BCE:', [y for y in time_years if y <= 0])
print()
# CE steps — show structure by grouping
ce_years = [y for y in time_years if y > 0]
print('CE:', ce_years)

# (b) Open each file and report variable name, units, shape
print()
print('=== Per-file variable inventory ===')
FILES = ['cropland', 'grazing_land', 'urban_area', 'population_density', 'total_rice']
file_info = {}
for name in FILES:
    d = xr.open_dataset(HYDE / f'{name}.nc')
    for var in d.data_vars:
        attrs = dict(d[var].attrs)
        print(f'  {name}.nc  →  var="{var}"  units="{attrs.get("units","?")}"  '
              f'long_name="{attrs.get("long_name","?")}"')
        file_info[name] = {'var': var, 'units': attrs.get('units','?')}
    d.close()


=== All 128 time steps (year, astronomical) ===
BCE: [-10000, -9000, -8000, -7000, -6000, -5000, -4000, -3000, -2000, -1000, 0]

CE: [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1710, 1720, 1730, 1740, 1750, 1760, 1770, 1780, 1790, 1800, 1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

=== Per-file variable inventory ===
  cropland.nc  →  var="cropland"  units="km**2"  long_name="Area cropland"
  grazing_land.nc  →  var="grazing_land"  units="km**2"  long

In [3]:
# Cell 3 — Per-variable, per-epoch summary statistics
#
# For each of the 8 target epochs and 5 variables: compute
#   mean, median, p95, p99, % zero (value == 0), % null (NaN)
# using lazy xarray slicing + numpy on the extracted slice only.
#
# population_density (capita/km²) is treated the same as the km² variables
# for distribution purposes — noted separately in output.
#
# Caution: each .values call loads one (2160×4320) slice into memory (~37MB
# at float32). We load one at a time and discard. Do not load the full array.

import pandas as pd

EPOCHS = [-8000, -4000, -1000, 0, 1000, 1500, 1900, 2000]

# Epoch labels for display (convert astronomical year 0 → "1 BCE")
def epoch_label(y):
    if y < 0:  return f'{abs(y)} BCE'
    if y == 0: return '1 BCE'
    return f'{y} CE'

# Map each filename to its internal variable name (confirmed in cell 2)
VAR_MAP = {
    'cropland':           'cropland',
    'grazing_land':       'grazing_land',
    'urban_area':         'urban_area',
    'population_density': 'population_density',
    'total_rice':         'total_rice',
}

UNITS = {
    'cropland':           'km²',
    'grazing_land':       'km²',
    'urban_area':         'km²',
    'population_density': 'cap/km²',
    'total_rice':         'km²',
}

rows = []
for fname, varname in VAR_MAP.items():
    ds_v = xr.open_dataset(HYDE / f'{fname}.nc')
    time_years_v = [t.year for t in ds_v['time'].values]

    for epoch in EPOCHS:
        t_idx = time_years_v.index(epoch)
        # Load only this one time slice — ~37MB
        arr = ds_v[varname].isel(time=t_idx).values.astype(np.float32)

        n_total = arr.size
        n_null  = int(np.isnan(arr).sum())
        valid   = arr[~np.isnan(arr)]
        n_zero  = int((valid == 0).sum())
        n_valid = len(valid)

        rows.append({
            'variable':  fname,
            'units':     UNITS[fname],
            'epoch':     epoch,
            'label':     epoch_label(epoch),
            'n_total':   n_total,
            'pct_null':  round(n_null  / n_total * 100, 2),
            'pct_zero':  round(n_zero  / n_valid * 100, 2) if n_valid > 0 else None,
            'mean':      round(float(np.mean(valid)),        4) if n_valid > 0 else None,
            'median':    round(float(np.median(valid)),      4) if n_valid > 0 else None,
            'p95':       round(float(np.percentile(valid, 95)), 4) if n_valid > 0 else None,
            'p99':       round(float(np.percentile(valid, 99)), 4) if n_valid > 0 else None,
            'max':       round(float(np.max(valid)),         4) if n_valid > 0 else None,
        })
        print(f'  {fname:20s}  {epoch_label(epoch):10s}  '
              f'%zero={n_zero/n_valid*100:.1f}%  mean={np.mean(valid):.4f}  '
              f'p99={np.percentile(valid,99):.4f}')

    ds_v.close()

summary = pd.DataFrame(rows)
summary.to_csv(OUT / '08_epoch_summary.csv', index=False)
print(f'\nSaved 08_epoch_summary.csv  ({len(summary)} rows)')


  total_rice            1900 CE     %zero=98.8%  mean=0.1788  p99=1.6449
  total_rice            2000 CE     %zero=98.5%  mean=0.5055  p99=20.9061

Saved 08_epoch_summary.csv  (40 rows)


In [4]:
# Cell 4 — Temporal trajectory: % zero and mean across all 8 epochs
#
# Two panels per variable: (top) % zero cells over time — shows when the
# signal emerges from silence; (bottom) mean value — shows intensity growth.
# These answer the key Task 8 question: when does anthropogenic land use
# become a non-trivial global signal?
#
# x-axis: epoch labels in chronological order.
# Population density plotted separately (different units, different story).

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

summary = pd.read_csv(OUT / '08_epoch_summary.csv')

# Chronological epoch order for x-axis
EPOCH_ORDER = [-8000, -4000, -1000, 0, 1000, 1500, 1900, 2000]
XLABELS = ['8000\nBCE', '4000\nBCE', '1000\nBCE', '1 BCE',
           '1000\nCE', '1500\nCE', '1900\nCE', '2000\nCE']

# Split variables: km² area vars vs. population density
AREA_VARS  = ['cropland', 'grazing_land', 'urban_area', 'total_rice']
POP_VARS   = ['population_density']

COLORS = {
    'cropland':           'steelblue',
    'grazing_land':       'darkorange',
    'urban_area':         'firebrick',
    'total_rice':         'seagreen',
    'population_density': 'purple',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('HYDE 3.4 — Signal emergence across epochs\n(% zero cells and mean value)', fontsize=12)

# Panel A: % zero for area variables (top-left)
ax = axes[0, 0]
for var in AREA_VARS:
    sub = summary[summary.variable == var].set_index('epoch').reindex(EPOCH_ORDER)
    ax.plot(range(len(EPOCH_ORDER)), sub['pct_zero'].values,
            marker='o', label=var, color=COLORS[var], linewidth=1.8)
ax.set_xticks(range(len(EPOCH_ORDER)))
ax.set_xticklabels(XLABELS, fontsize=8)
ax.set_ylabel('% zero cells')
ax.set_title('% zero — area variables (km²)', fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(0, 102)
ax.axhline(95, color='gray', lw=0.6, ls='--', label='95% zero')

# Panel B: % zero for population density (top-right)
ax = axes[0, 1]
sub = summary[summary.variable == 'population_density'].set_index('epoch').reindex(EPOCH_ORDER)
ax.plot(range(len(EPOCH_ORDER)), sub['pct_zero'].values,
        marker='o', color=COLORS['population_density'], linewidth=1.8)
ax.set_xticks(range(len(EPOCH_ORDER)))
ax.set_xticklabels(XLABELS, fontsize=8)
ax.set_ylabel('% zero cells')
ax.set_title('% zero — population density (cap/km²)', fontsize=10)
ax.set_ylim(0, 102)
ax.axhline(95, color='gray', lw=0.6, ls='--')

# Panel C: mean value for area variables (bottom-left) — log scale
ax = axes[1, 0]
for var in AREA_VARS:
    sub = summary[summary.variable == var].set_index('epoch').reindex(EPOCH_ORDER)
    vals = sub['mean'].values.astype(float)
    vals = np.where(vals <= 0, np.nan, vals)
    ax.plot(range(len(EPOCH_ORDER)), vals,
            marker='o', label=var, color=COLORS[var], linewidth=1.8)
ax.set_xticks(range(len(EPOCH_ORDER)))
ax.set_xticklabels(XLABELS, fontsize=8)
ax.set_yscale('log')
ax.set_ylabel('Mean value (km², log scale)')
ax.set_title('Mean — area variables (log scale)', fontsize=10)
ax.legend(fontsize=8)

# Panel D: mean population density (bottom-right) — linear
ax = axes[1, 1]
sub = summary[summary.variable == 'population_density'].set_index('epoch').reindex(EPOCH_ORDER)
ax.plot(range(len(EPOCH_ORDER)), sub['mean'].values,
        marker='o', color=COLORS['population_density'], linewidth=1.8)
ax.set_xticks(range(len(EPOCH_ORDER)))
ax.set_xticklabels(XLABELS, fontsize=8)
ax.set_ylabel('Mean (cap/km²)')
ax.set_title('Mean — population density (linear scale)', fontsize=10)

plt.tight_layout()
plt.savefig(OUT / '08_signal_emergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 08_signal_emergence.png')


Saved 08_signal_emergence.png


In [5]:
# Cell 5 — Baseline period for "pre-anthropogenic" land use
#
# Task 8 question: what epoch is defensible as a baseline representing
# absence of significant anthropogenic modification, against which a
# query window's land use can be reported as an anomaly?
#
# Approach: for cropland and grazing (the two meaningful land-use variables),
# compute the ratio of each CE epoch's mean to each candidate baseline epoch.
# A ratio of 1.0 = no change from baseline; 10 = 10x above baseline.
# Then identify the % of cells that cross a "non-trivial" threshold (> 1 km²)
# at each epoch — this gives a spatial sense of when land use becomes widespread
# enough to matter for a typical basin query.
#
# Candidate baselines: 8000 BCE (near-zero globally), 4000 BCE (pastoralism
# emerging), 1000 BCE (agriculture established in core regions).

import numpy as np

CANDIDATE_BASELINES = [-8000, -4000, -1000]
CE_EPOCHS           = [1000, 1500, 1900, 2000]
VARS                = ['cropland', 'grazing_land']
NONTRIVIAL_KM2      = 1.0   # threshold for "meaningful land use in a cell"

# Load all needed slices: baselines + CE epochs, for cropland and grazing
slices = {}
for fname in VARS:
    ds_v = xr.open_dataset(HYDE / f'{fname}.nc')
    time_years_v = [t.year for t in ds_v['time'].values]
    for epoch in CANDIDATE_BASELINES + CE_EPOCHS:
        arr = ds_v[fname].isel(time=time_years_v.index(epoch)).values.astype(np.float32)
        slices[(fname, epoch)] = arr
    ds_v.close()
    print(f'Loaded {fname}: {len(CANDIDATE_BASELINES + CE_EPOCHS)} epochs')

# --- Table 1: ratio of CE epoch mean to each baseline mean ---
print('\n=== Mean value ratio: CE epoch / baseline (global non-null cells) ===')
header = f'{"Variable":<15}  {"Baseline":<12}  ' + ''.join(f'{str(e)+" CE":>10}  ' for e in CE_EPOCHS)
print(header)
print('-' * 75)

ratio_rows = []
for fname in VARS:
    for base in CANDIDATE_BASELINES:
        base_arr   = slices[(fname, base)]
        base_valid = base_arr[~np.isnan(base_arr)]
        base_mean  = np.mean(base_valid)
        label      = f'{abs(base)} BCE'
        row_str    = f'{fname:<15}  {label:<12}  '
        for e in CE_EPOCHS:
            ce_arr   = slices[(fname, e)]
            ce_valid = ce_arr[~np.isnan(ce_arr)]
            ce_mean  = np.mean(ce_valid)
            ratio    = ce_mean / base_mean if base_mean > 0 else np.nan
            row_str += f'{ratio:>10.1f}  '
            ratio_rows.append({'variable': fname, 'baseline': base,
                                'ce_epoch': e, 'ratio': round(ratio, 2)})
        print(row_str)

# --- Table 2: % of cells above non-trivial threshold at each epoch ---
print(f'\n=== % cells with > {NONTRIVIAL_KM2} km² land use (spatial footprint of meaningful activity) ===')
all_epochs = CANDIDATE_BASELINES + CE_EPOCHS
header2 = f'{"Variable":<15}  ' + ''.join(
    f'{(str(abs(e))+"BCE" if e<=0 else str(e)+"CE"):>9}  ' for e in all_epochs)
print(header2)
print('-' * 90)

for fname in VARS:
    row_str = f'{fname:<15}  '
    for e in all_epochs:
        arr   = slices[(fname, e)]
        valid = arr[~np.isnan(arr)]
        pct   = (valid > NONTRIVIAL_KM2).mean() * 100
        row_str += f'{pct:>8.1f}%  '
    print(row_str)

pd.DataFrame(ratio_rows).to_csv(OUT / '08_baseline_ratios.csv', index=False)
print('\nSaved 08_baseline_ratios.csv')


cropland              0.0%       0.9%       4.8%      10.8%      13.9%      25.9%      33.2%  
grazing_land          0.1%       2.7%       7.4%      19.7%      23.2%      45.2%      49.3%  

Saved 08_baseline_ratios.csv
